In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

from fla.layers import GatedDeltaNet

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

这个似乎需要linux才行

In [ ]:
def _gated_delta_head_shapes(hidden_size: int, num_heads: int = 6):
    """fla GatedDeltaNet 在 use_gate=True 时要求 num_heads * head_dim == 0.75 * hidden_size（均为整数）。"""
    key_dim = (hidden_size * 3) // 4
    if key_dim * 4 != hidden_size * 3:
        raise ValueError("hidden_size（即 d_model）必须被 4 整除，以满足 0.75*hidden_size 为整数")
    if key_dim % num_heads != 0:
        raise ValueError(f"请调整 num_heads 使其整除 key_dim={key_dim}")
    head_dim = key_dim // num_heads
    return num_heads, head_dim


class PatchEmbed(nn.Module):
    """图像分块嵌入 + 可学习位置编码。输入 [B,3,H,W]，输出 [B,L,d_model]。"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, d_model=512):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(
            in_chans, d_model, kernel_size=patch_size, stride=patch_size, bias=False
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x + self.pos_embed


class GDN(nn.Module):
    """Patch 嵌入 + 多层 `fla.layers.GatedDeltaNet` + 均值池化 + 分类头。"""

    def __init__(
        self,
        num_classes,
        img_size=224,
        patch_size=16,
        in_chans=3,
        d_model=512,
        num_layers=6,
        dropout=0.1,
        num_heads=6,
        mode="chunk",
        use_short_conv=True,
    ):
        super().__init__()
        num_heads, head_dim = _gated_delta_head_shapes(d_model, num_heads=num_heads)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, d_model)
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [
                GatedDeltaNet(
                    hidden_size=d_model,
                    head_dim=head_dim,
                    num_heads=num_heads,
                    mode=mode,
                    use_short_conv=use_short_conv,
                    layer_idx=i,
                )
                for i in range(num_layers)
            ]
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.pos_drop(self.patch_embed(x))
        for blk in self.blocks:
            x = blk(x)[0]
        x = self.norm(x).mean(dim=1)
        return self.head(x)


def build_model(num_classes, **kwargs):
    return GDN(num_classes=num_classes, **kwargs).to(device)

In [ ]:
def stratified_split(dataset, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, random_seed=42):
    targets = np.array(dataset.targets)
    classes = np.unique(targets)
    train_idx, val_idx, test_idx = [], [], []
    rng = np.random.default_rng(random_seed)
    for cls in classes:
        cls_indices = np.where(targets == cls)[0]
        rng.shuffle(cls_indices)
        n_cls = len(cls_indices)
        n_train = int(round(train_ratio * n_cls))
        n_val = int(round(val_ratio * n_cls))
        n_test = n_cls - n_train - n_val
        if n_test < 0:
            n_test = 0
            n_train = n_cls - n_val
        train_idx.extend(cls_indices[:n_train])
        val_idx.extend(cls_indices[n_train : n_train + n_val])
        test_idx.extend(cls_indices[n_train + n_val :])
    return train_idx, val_idx, test_idx


class SubsetWithTransform(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        if self.transform:
            x = self.transform(x)
        if not isinstance(x, torch.Tensor):
            from torchvision.transforms import functional as F

            x = F.to_tensor(x)
        return x, y


def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
        pbar.set_postfix(Loss=f"{loss.item():.4f}", Acc=f"{correct/total:.4f}")
    return running_loss / total, correct / total


def validate(model, dataloader, criterion, device, epoch, phase="Val"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_pred, all_labels = [], []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [{phase}]", leave=False)
    with torch.no_grad():
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            all_pred.append(preds.cpu())
            all_labels.append(labels.cpu())
            pbar.set_postfix(Loss=f"{loss.item():.4f}", Acc=f"{correct/total:.4f}")
    y_p = torch.cat(all_pred).numpy()
    y_t = torch.cat(all_labels).numpy()
    macro_f1 = f1_score(y_t, y_p, average="macro")
    return running_loss / total, correct / total, macro_f1


def plot_curves(train_losses, val_losses, train_accs, val_accs, val_f1s, save_path="training_curves_gdn.png"):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(epochs, train_losses, label="Train loss")
    plt.plot(epochs, val_losses, label="Val loss")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)
    plt.subplot(1, 3, 2)
    plt.plot(epochs, train_accs, label="Train acc")
    plt.plot(epochs, val_accs, label="Val acc")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)
    plt.subplot(1, 3, 3)
    plt.plot(epochs, val_f1s, label="Val macro F1")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()
    print(f"曲线已保存: {save_path}")

In [ ]:
def main():
    for _base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        _src = _base / "src"
        if (_src / "raicom").is_dir() and str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
            break
    from raicom.paths import default_data_root

    data_root = os.environ.get("RAICOM_DATA_ROOT", default_data_root())
    batch_size = 16
    epochs = 50
    lr = 3e-4
    weight_decay = 0.05
    ckpt_path = Path.cwd() / "gdn_best.pth"

    transform_train = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )
    transform_eval = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    full_dataset = datasets.ImageFolder(root=data_root, transform=None)
    num_classes = len(full_dataset.classes)
    print(f"类别数 {num_classes}: {full_dataset.classes}")
    print(f"总样本 {len(full_dataset)}, 数据根目录 {data_root}")

    train_idx, val_idx, test_idx = stratified_split(full_dataset, random_seed=42)
    train_ds = SubsetWithTransform(full_dataset, train_idx, transform_train)
    val_ds = SubsetWithTransform(full_dataset, val_idx, transform_eval)
    test_ds = SubsetWithTransform(full_dataset, test_idx, transform_eval)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    model = build_model(
        num_classes,
        d_model=512,
        num_layers=6,
        dropout=0.1,
        mode="chunk",
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    val_f1s = []
    best_acc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        va_loss, va_acc, va_f1 = validate(model, val_loader, criterion, device, epoch, "Val")
        scheduler.step()
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        train_accs.append(tr_acc)
        val_accs.append(va_acc)
        val_f1s.append(va_f1)
        print(
            f"Epoch {epoch:03d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f} f1 {va_f1:.4f}"
        )
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> 保存最佳权重 {ckpt_path} (val_acc={va_acc:.4f})")

    if ckpt_path.is_file():
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        print("警告: 未写入 checkpoint（验证集从未提升），使用最后一轮权重做测试")
    te_loss, te_acc, te_f1 = validate(model, test_loader, criterion, device, 0, "Test")
    print(f"测试集: loss={te_loss:.4f} acc={te_acc:.4f} macro_f1={te_f1:.4f}")
    plot_curves(train_losses, val_losses, train_accs, val_accs, val_f1s, save_path="training_curves_gdn.png")

In [ ]:
main()